# Deal Postmortem voice rendering
Choose **Runtime → Change runtime type → GPU**. Run cells in order. Upload only a recording you own or have permission to clone. Colab storage is temporary; download the output ZIP before the session ends.

In [ ]:
import subprocess
from pathlib import Path

repo = Path('/content/peregrine')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/oscarhugs/peregrine.git', str(repo)], check=True)
channel = repo / 'MA Channel'
subprocess.run(['python', '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
subprocess.run(['uv', 'venv', '/content/voice-venv', '--python', '3.11'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', '/content/voice-venv/bin/python', '-r', str(channel / 'tools/voice/requirements.txt')], check=True)
subprocess.run(['/content/voice-venv/bin/python', '-c', 'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; print(torch.cuda.get_device_name(0))'], check=True)


Set `slug` to `_test` for the included two-section sample, or to your approved video folder name. Upload `me_full.wav`. For a real video, also upload its approved `script.md`.

In [ ]:
from google.colab import files

slug = '_test'  # e.g. '001-red-lobster'
uploads = files.upload()
assert 'me_full.wav' in uploads, 'Upload me_full.wav'
reference_dir = channel / 'tools/voice/reference'
reference_dir.mkdir(parents=True, exist_ok=True)
(reference_dir / 'me_full.wav').write_bytes(uploads['me_full.wav'])
script = channel / 'videos' / slug / 'script.md'
if 'script.md' in uploads:
    script.parent.mkdir(parents=True, exist_ok=True)
    script.write_bytes(uploads['script.md'])
assert script.exists(), f'Upload script.md for {slug}'
subprocess.run(['/content/voice-venv/bin/python', str(channel / 'tools/voice/prepare_reference.py')], cwd=channel, check=True)
subprocess.run(['/content/voice-venv/bin/python', str(channel / 'tools/voice/voice.py'), str(script), '--dry-run'], cwd=channel, check=True)


In [ ]:
import shutil

subprocess.run(['/content/voice-venv/bin/python', str(channel / 'tools/voice/voice.py'), str(script)], cwd=channel, check=True)
manifest = script.parent / 'vo' / 'manifest.json'
assert manifest.exists(), 'Narration manifest was not created'
archive = shutil.make_archive('/content/voice-output', 'zip', root_dir=script.parent, base_dir='vo')
print(f'Created {archive}')
files.download(archive)
